In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import json

import sys

sys.path.append("../")

##################################################################
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_VISIBLE_DEVICES"] = "3,4,5,6"
##################################################################

import logging
from src.utils import logging_utils
from src.utils import env_utils
import torch
import transformers
from rich.console import Console
from rich.theme import Theme

custom_theme = Theme({"default": "white"})
console = Console(theme=custom_theme)


# Override print globally
def rich_print(*args, **kwargs):
    console.print(*args, **kwargs)


logger = logging.getLogger(__name__)

logging.basicConfig(
    level=logging.DEBUG,
    format=logging_utils.DEFAULT_FORMAT,
    datefmt=logging_utils.DEFAULT_DATEFMT,
    stream=sys.stdout,
)

logger.info(f"{torch.__version__=}, {torch.version.cuda=}")
logger.info(
    f"{torch.cuda.is_available()=}, {torch.cuda.device_count()=}, {torch.cuda.get_device_name()=}"
)
logger.info(f"{transformers.__version__=}")

2026-02-25 17:31:30 __main__ INFO     torch.__version__='2.10.0+cu128', torch.version.cuda='12.8'
2026-02-25 17:31:30 __main__ INFO     torch.cuda.is_available()=True, torch.cuda.device_count()=4, torch.cuda.get_device_name()='NVIDIA A100 80GB PCIe'
2026-02-25 17:31:30 __main__ INFO     transformers.__version__='5.1.0'


## Loading the LM

In [3]:
from src.models import ModelandTokenizer

# model_key = "allenai/Olmo-3-1125-32B"
# model_key = "allenai/Olmo-3.1-32B-Instruct"
# model_key = "allenai/Olmo-3.1-7B-RL-Zero-Code"

# model_key = "meta-llama/Llama-3.3-70B-Instruct"

# model_key = "deepseek-ai/deepseek-coder-33b-base"
# model_key = "deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct"

# model_key = "openai/gpt-oss-20b"
# model_key = "openai/gpt-oss-120b" #! weird results. known issue.

# model_key = "Qwen/Qwen3-Coder-30B-A3B-Instruct"
# model_key = "Qwen/Qwen2.5-Coder-14B"
# model_key = "Qwen/Qwen2.5-Coder-14B-Instruct"
# model_key = "Qwen/Qwen2.5-Coder-32B"
model_key = "Qwen/Qwen2.5-Coder-32B-Instruct"
# model_key = "Qwen/Qwen2.5-72B"
# model_key = "Qwen/Qwen2.5-72B-Instruct"

# model_key = "bigcode/starcoder2-15b"

# model_key = "zai-org/GLM-4.7-Flash"

mt = ModelandTokenizer(
    model_key=model_key,
    dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)

2026-02-25 17:31:34 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory
2026-02-25 17:31:34 httpcore.connection DEBUG    connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=10 socket_options=None
2026-02-25 17:31:34 httpcore.connection DEBUG    connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7f495fabaa50>
2026-02-25 17:31:34 httpcore.connection DEBUG    start_tls.started ssl_context=<ssl.SSLContext object at 0x7f495f45d350> server_hostname='huggingface.co' timeout=10
2026-02-25 17:31:34 httpcore.connection DEBUG    start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7f495f629820>
2026-02-25 17:31:34 httpcore.http11 DEBUG    send_request_headers.started request=<Request [b'HEAD']>
2026-02-25 17:31:34 httpcore.http11 DEBUG    send_request_headers.complete
2026-02-25 17:31:34 http

Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

2026-02-25 17:32:00 httpcore.connection DEBUG    close.started
2026-02-25 17:32:00 httpcore.connection DEBUG    close.complete
2026-02-25 17:32:00 httpcore.connection DEBUG    connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=10 socket_options=None
2026-02-25 17:32:00 httpcore.connection DEBUG    connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7f495ddc9550>
2026-02-25 17:32:00 httpcore.connection DEBUG    start_tls.started ssl_context=<ssl.SSLContext object at 0x7f495f45d350> server_hostname='huggingface.co' timeout=10
2026-02-25 17:32:00 httpcore.connection DEBUG    start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7f495c312bd0>
2026-02-25 17:32:00 httpcore.http11 DEBUG    send_request_headers.started request=<Request [b'HEAD']>
2026-02-25 17:32:00 httpcore.http11 DEBUG    send_request_headers.complete
2026-02-25 17:32:00 httpcore.http11 DEBUG    send_request_body.started request=<Request [b'HE

## Debug

In [ ]:
from src.goals.eval import run_evaluation, pretty_print_eval_results
from src.goals.predicate_data import MultiPredicateDataset


PREDICATE_IDX = 5

eval_dataset = MultiPredicateDataset.load_dataset()

target_predicate = eval_dataset[PREDICATE_IDX]
rich_print(target_predicate)

In [ ]:
from src.tokens import prepare_input
from src.goals.data import CodeGenerations, Code
from src.goals.functional import supports_chat_template, apply_chat_template
from src.functional import generate_with_patch

SAMPLE_IDX = 2
sample = target_predicate.samples[SAMPLE_IDX]
instruction = sample.instruction
unfinished_code = sample.code
function_name = sample.function_name or "func"
unit_tests = sample.unit_tests
prefills = []
for ptype, pfill_list in sample.prefills.items():
    prefills.extend([prefill, ptype] for prefill in pfill_list)


if supports_chat_template(mt):
    prompts = [
        apply_chat_template(
            mt=mt,
            user_query=instruction.strip(),
            incomplete_code=unfinished_code + prefill,
        )
        for prefill, ptype in prefills
    ]
else:
    prompts = [
        instruction + "<code>" + unfinished_code + prefill
        for (prefill, ptype) in prefills
    ]


inputs = prepare_input(
    prompts=prompts,
    tokenizer=mt,
)

output = generate_with_patch(
    mt=mt,
    inputs=inputs,
    n_gen_per_prompt=1,
    max_new_tokens=64,
    do_sample=False,
    remove_prefix=True,
)

results = CodeGenerations(
    sample=sample,
    generations=[
        Code(prefill=prefill, prefill_type=ptype, full_prompt=inp, lm_generation=out)
        for (prefill, ptype), inp, out in zip(prefills, prompts, output)
    ],
)

# print(f"[white]{str(results)}[/white]")

eval_results = run_evaluation(
    result = results,
    function_name=function_name,
    code_stop_tag="</code>",
)

# Print results in tabular format
pretty_print_eval_results(eval_results)

In [ ]:
prefill_idx = -1
generated_code = eval_results.generations[prefill_idx]
print(generated_code)

print(f'Full Prompt:\n"{generated_code.full_prompt}"\n')
print(f'Full Output:\n"{generated_code.lm_generation}"\n')

In [ ]:
print(target_predicate.samples[2])

In [ ]:
from src.goals.sampling import MCTSSearch

full_prompt = prompts[prefill_idx]
print(f'Full Prompt for MCTS:\n"{full_prompt}"\n')

search = MCTSSearch(
    mt=mt,
    prompt=full_prompt,
    max_depth=32, top_k=5,
    prob_threshold=0.01,
    temperature=1.0,
    c_puct=1.0,
    min_children_up_to_depth=3,
    min_children=2,
)

root = search.run(
    n_iterations=1500,
    depth_first=True,
)

In [ ]:
print(search.get_stats())

paths = search.get_all_paths(
    # terminal_only=True, 
    sort_by="prob", 
    # contains="else False"
    # depth=5
)

# Print results
for p in paths:
    print(
        f'<v={p["visits"]}, p={p["prob"]:.4f}> {repr(p["text"])}'
    )

In [ ]:
# Inspect results
search.print_tree(max_depth=None, show_colors=False)

In [ ]:
from src.functional import predict_next_token

prefix = sample.code + generated_code.prefill
print(prefix)

test = ' == n % 2)'

test_prompt = generated_code.full_prompt + test
print(test_prompt)

predict_next_token(
    mt=mt,
    inputs = test_prompt
)

In [ ]:
# node = search.find_node_by_text(' == n % 2)')
# node

In [ ]:
# # Debug why it's not being expanded
# if node:
#     print(search.debug_node(node))

#     # If it's not terminal, see what children would be created
#     if not node.is_terminal and node.is_leaf():
#         search.force_expand(node)

#     # Check PUCT scores at its parent to see why siblings might be selected first
#     if node.parent:
#         for child, score in search.get_puct_scores(node.parent):
#             print(f"{child.token_str!r}: PUCT={score:.4f}, visits={child.visits}")

In [ ]:
from src.goals.eval import evaluate_mcts_tree, clear_mcts_colors

prefix = sample.code + generated_code.prefill
clear_mcts_colors(search.root)

counts = evaluate_mcts_tree(
    root=search.root,
    prefix=prefix,
    unit_tests=sample.unit_tests,
    function_name=sample.function_name or "func",
    terminal_only=True
)

print(counts)

In [ ]:
search.print_tree(max_depth=None, show_colors=True)

In [ ]:
search.save_tree("tree.html", show_colors=True)

## Checking the green solutions

In [ ]:
# def get_green_solutions(root, prefix, tokenizer=None):                                                                                                                                                                        
#     """Return list of (full_code, eval_result) for all green leaf nodes."""                                                                                                                                                   
#     results = []                                                                                                                                                                                                              
#     def walk(node):                                                                                                                                                                                                           
#         if node.is_leaf() and node.color == "green":                                                                                                                                                                        
#             results.append((prefix + node.get_path_text(), node.eval_result))
#         for child in node.children:
#             walk(child)
#     walk(root)
#     return results

# # After evaluate_mcts_tree(root, prefix, unit_tests, function_name):
# solutions = get_green_solutions(root, prefix="")
# for code, eval_result in solutions:
#     print(code)
#     print("---")

In [ ]:
def get_green_solutions_by_first_token(root, prefix):
    """Group green solutions by their first token (reveals which predicate was pursued first)."""
    from collections import defaultdict
    groups = defaultdict(list)
    def walk(node):
        if node.is_leaf() and node.color == "green":
            # Walk up to find the root's direct child (first token)
            n = node
            while n.parent and not n.parent.is_root():
                n = n.parent
            first_tok = n.token_str
            groups[first_tok].append(prefix + node.get_path_text())
        for child in node.children:
            walk(child)
    walk(root)
    return dict(groups)

groups = get_green_solutions_by_first_token(root, prefix="")
for first_tok, codes in groups.items():
    print(f"First token: {first_tok!r}")
    for code in codes:
        print(f'"{code}"')
        # print("---")

## Build Trees for the Target Predicate

In [4]:
from src.goals.predicate_data import MultiPredicateDataset

PREDICATE_IDX = 5

eval_dataset = MultiPredicateDataset.load_dataset()
target_predicate = eval_dataset[PREDICATE_IDX]
rich_print(target_predicate)

2026-02-25 17:33:04 src.goals.predicate_data INFO     Loaded TargetPredicate 'has_unique_elements' with 3 sample(s) from /disk/u/arnab/Codes/Projects/goals/data/multi_predicate/has_unique_elements.json
2026-02-25 17:33:04 src.goals.predicate_data INFO     Loaded TargetPredicate 'is_anagram' with 4 sample(s) from /disk/u/arnab/Codes/Projects/goals/data/multi_predicate/is_anagram.json
2026-02-25 17:33:04 src.goals.predicate_data INFO     Loaded TargetPredicate 'is_palindrome' with 5 sample(s) from /disk/u/arnab/Codes/Projects/goals/data/multi_predicate/is_palindrome.json
2026-02-25 17:33:04 src.goals.predicate_data INFO     Loaded TargetPredicate 'is_perfect_square' with 4 sample(s) from /disk/u/arnab/Codes/Projects/goals/data/multi_predicate/is_perfect_square.json
2026-02-25 17:33:04 src.goals.predicate_data INFO     Loaded TargetPredicate 'is_power_of_two' with 3 sample(s) from /disk/u/arnab/Codes/Projects/goals/data/multi_predicate/is_power_of_two.json
2026-02-25 17:33:04 src.goals.pr

TargetPredicate(
    p_plus=PredicateSpec(
        description='check whether lst is sorted in non-decreasing order (lst == sorted(lst))',
        anchor_tokens=[
            '< lst[i]',
            '<= lst[',
            '<= lst[i',
            '<= lst[i+1]',
            '== sorted',
            '== sorted(',
            'a < b',
            'a == b',
            'all(',
            'all(i',
            'all(lst',
            'all(x',
            'earlier < later',
            'i + 1]',
            'i < j',
            'i == j',
            'i+1]',
            'lst ==',
            'lst[1:]',
            'lst[i +',
            'lst[i+1]',
            'lst[i-1] < lst[i]',
            'lst[i-1] <=',
            'lst[i-1] <= lst[i]',
            'lst[i]',
            'lst[i] <',
            'lst[i] <=',
            'lst[i] <= lst[i+1]',
            'lst[i] >=',
            'sorted(',
            'sorted(lst)',
            'x < y',
            'x <= y',
            'x == y',
            'zip(lst'
        ]
    ),
    samples=[
        MultiPredicateSample(
            instruction='Write a Python function that returns True iff the list has at least 2 elements and is 
sorted in non-decreasing order.\n\n',
            code='def is_long_sorted(lst: list) -> bool:\n    """Return True iff lst has at least 2 elements and is
sorted in non-decreasing order."""\n    return ',
            unit_tests=[
                ([1, 2, 3], True),
                ([3, 2, 1], False),
                ([1], False),
                ([], False),
                ([1, 1, 2], True),
                ([2, 1], False),
                ([5, 5], True),
                ([-1, 0, 1], True),
                ([0, 0], True)
            ],
            prefills={
                'aligned_p_plus': ['sorted(', 'lst ==', 'lst == sorted('],
                'aligned_p_minus': ['len(', 'len(lst)'],
                'adversarial': ['not (', 'False if', 'True if not'],
                'inconsequential': ['']
            },
            function_name='is_long_sorted',
            p_minus=PredicateSpec(
                description='check whether len(lst) >= 2',
                anchor_tokens=[
                    '1 < len',
                    '2 <=',
                    '< len(',
                    '<= len(',
                    '> 1',
                    '>= 2',
                    'len(',
                    'len(lst)',
                    'len(lst) >',
                    'len(lst) > 1',
                    'len(lst) >=',
                    'len(lst) >= 2'
                ]
            )
        ),
        MultiPredicateSample(
            instruction='Write a Python function that returns True iff the list is non-empty, its first element is 
non-negative, and it is sorted in non-decreasing order.\n\n',
            code='def is_nonneg_sorted(lst: list) -> bool:\n    """Return True iff lst is non-empty, starts with a 
non-negative value, and is sorted in non-decreasing order."""\n    return ',
            unit_tests=[
                ([0, 1, 2], True),
                ([-1, 0, 1], False),
                ([1, 3, 5], True),
                ([2, 1], False),
                ([], False),
                ([0], True),
                ([-2, -1, 0], False),
                ([1, 1, 2], True),
                ([-1], False)
            ],
            prefills={
                'aligned_p_plus': ['sorted(', 'lst ==', 'lst == sorted('],
                'aligned_p_minus': ['lst[0]', 'lst[0] >='],
                'adversarial': ['not (', 'False if', 'True if not'],
                'inconsequential': ['']
            },
            function_name='is_nonneg_sorted',
            p_minus=PredicateSpec(
                description='check whether lst is non-empty and lst[0] >= 0',
                anchor_tokens=['>= 0', 'bool(lst)', 'lst) and lst[0]', 'lst[0] >=', 'lst[0] >= 0']
            )
        ),
        MultiPredicateSample(
            instruction='Write a Python function that returns True iff t

In [6]:
from src.goals.predicate_sampling import (
    MCTSRunResult,
    MCTSSearchResults_for_MultiPredicateSample,
    MCTSSearchResults_for_TargetPredicate,
)
from src.goals.predicate_data import MultiPredicateSample
from src.goals.sampling import MCTSSearch
from src.goals.eval import evaluate_mcts_tree, clear_mcts_colors
from src.goals.anchor_extraction import get_green_completions
from src.goals.functional import supports_chat_template, apply_chat_template


def build_full_prompt(sample: MultiPredicateSample, prefill: str, mt) -> str:
    if supports_chat_template(mt):
        return apply_chat_template(
            mt=mt,
            user_query=sample.instruction.strip(),
            incomplete_code=sample.code + prefill,
        )
    return sample.instruction + "<code>" + sample.code + prefill


MCTS_CONFIG = dict(
    n_iterations=1500,
    max_depth=32,
    top_k=5,
    prob_threshold=0.01,
    temperature=1.0,
    c_puct=1.0,
    min_children_up_to_depth=3,
    min_children=2,
    depth_first=True,
)

In [7]:
import os

target_predicate = eval_dataset[PREDICATE_IDX]
model_tag = mt.name.split("/")[-1]
SAVE_PATH = os.path.join(
    env_utils.DEFAULT_RESULTS_DIR,
    "MultiPredicate_MCTS", 
    model_tag,
    f"{target_predicate.name}.json"
)

os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)

# ── Load cached results if available ─────────────────────────────────────────
if os.path.exists(SAVE_PATH):
    rich_print(f"[yellow]Loading cached results from {SAVE_PATH}[/yellow]")
    tp_results = MCTSSearchResults_for_TargetPredicate.load(SAVE_PATH)

# ── Otherwise run MCTS for every sample ──────────────────────────────────────
else:
    tp_results = MCTSSearchResults_for_TargetPredicate(
        target_predicate_name=target_predicate.name,
        model_name=mt.name,
        mcts_config=MCTS_CONFIG,
    )

    for sample_idx, p_neg_sample in enumerate(target_predicate.samples):
        rich_print(
            f"\n[bold]Sample {sample_idx + 1}/{len(target_predicate.samples)}[/bold]"
            f"  fn=[cyan]{p_neg_sample.function_name}[/cyan]"
            f"  p-=[italic]{p_neg_sample.p_minus.description if p_neg_sample.p_minus else '?'}[/italic]"
        )

        sample_result = MCTSSearchResults_for_MultiPredicateSample(sample=p_neg_sample)

        prefill = p_neg_sample.prefills["inconsequential"][0]
        prompt  = build_full_prompt(p_neg_sample, prefill, mt)
        prefix  = p_neg_sample.code + prefill

        # 1) MCTS search
        search = MCTSSearch(
            mt=mt,
            prompt=prompt,
            max_depth=MCTS_CONFIG["max_depth"],
            top_k=MCTS_CONFIG["top_k"],
            prob_threshold=MCTS_CONFIG["prob_threshold"],
            temperature=MCTS_CONFIG["temperature"],
            c_puct=MCTS_CONFIG["c_puct"],
            min_children_up_to_depth=MCTS_CONFIG["min_children_up_to_depth"],
            min_children=MCTS_CONFIG["min_children"],
        )
        root = search.run(
            n_iterations=MCTS_CONFIG["n_iterations"],
            depth_first=MCTS_CONFIG["depth_first"],
        )

        # 2) Evaluate tree (sets .color and .eval_result on leaves)
        counts = evaluate_mcts_tree(
            root=root,
            prefix=prefix,
            unit_tests=p_neg_sample.unit_tests,
            function_name=p_neg_sample.function_name or "func",
            terminal_only=True,
        )

        # 3) Extract unique green completions
        green_completions = sorted(set(
            c.strip() for c in get_green_completions(root, prefix) if c.strip()
        ))

        run = MCTSRunResult(
            prefill=prefill,
            prefill_type="inconsequential",
            prompt=prompt,
            prefix=prefix,
            mcts_config=MCTS_CONFIG,
            mcts_stats=search.get_stats(),
            eval_counts=counts,
            green_completions=green_completions,
        )
        sample_result.runs.append(run)
        sample_result._trees[prefix] = root          # live tree; lost on save/load

        tp_results.sample_results.append(sample_result)
        rich_print(f"  eval: {counts} | {len(green_completions)} unique green completions")

    tp_results.save(SAVE_PATH)
    rich_print(f"\n[green]Saved → {SAVE_PATH}[/green]")

# ── Summary ───────────────────────────────────────────────────────────────────
rich_print(f"\n[bold]Summary[/bold] — [cyan]{tp_results.target_predicate_name}[/cyan]  ({tp_results.model_name})")
for i, sr in enumerate(tp_results.sample_results):
    for run in sr.runs:
        rich_print(
            f"  sample {i} | prefill={run.prefill!r:6s} | {run.eval_counts}"
            f" | {len(run.green_completions)} green"
        )

Sample 1/4  fn=is_long_sorted  p-=check whether len(lst) >= 2

2026-02-25 17:34:00 src.goals.sampling DEBUG    MCTS iteration 50/1500, total nodes: 287
2026-02-25 17:34:02 src.goals.sampling DEBUG    MCTS iteration 100/1500, total nodes: 307
2026-02-25 17:34:02 src.goals.sampling DEBUG    MCTS iteration 150/1500, total nodes: 310
2026-02-25 17:34:02 src.goals.sampling DEBUG    MCTS iteration 200/1500, total nodes: 310
2026-02-25 17:34:05 src.goals.sampling DEBUG    MCTS iteration 250/1500, total nodes: 342
2026-02-25 17:34:05 src.goals.sampling DEBUG    MCTS iteration 300/1500, total nodes: 342
2026-02-25 17:34:05 src.goals.sampling DEBUG    MCTS iteration 350/1500, total nodes: 343
2026-02-25 17:34:05 src.goals.sampling DEBUG    MCTS iteration 400/1500, total nodes: 343
2026-02-25 17:34:05 src.goals.sampling DEBUG    MCTS iteration 450/1500, total nodes: 343
2026-02-25 17:34:05 src.goals.sampling DEBUG    MCTS iteration 500/1500, total nodes: 343
2026-02-25 17:34:05 src.goals.sampling DEBUG    MCTS iteration 550/1500, total nodes: 343
2026-02-25 

eval: {'green': 42, 'orange': 10, 'red': 3, 'total': 55} | 23 unique green completions

Sample 2/4  fn=is_nonneg_sorted  p-=check whether lst is non-empty and lst[0] >= 0

2026-02-25 17:34:58 src.goals.sampling DEBUG    MCTS iteration 50/1500, total nodes: 418
2026-02-25 17:34:58 src.goals.sampling DEBUG    MCTS iteration 100/1500, total nodes: 418
2026-02-25 17:34:58 src.goals.sampling DEBUG    MCTS iteration 150/1500, total nodes: 418
2026-02-25 17:34:58 src.goals.sampling DEBUG    MCTS iteration 200/1500, total nodes: 419
2026-02-25 17:34:58 src.goals.sampling DEBUG    MCTS iteration 250/1500, total nodes: 419
2026-02-25 17:34:58 src.goals.sampling DEBUG    MCTS iteration 300/1500, total nodes: 419
2026-02-25 17:35:04 src.goals.sampling DEBUG    MCTS iteration 350/1500, total nodes: 500
2026-02-25 17:35:04 src.goals.sampling DEBUG    MCTS iteration 400/1500, total nodes: 500
2026-02-25 17:35:05 src.goals.sampling DEBUG    MCTS iteration 450/1500, total nodes: 515
2026-02-25 17:35:11 src.goals.sampling DEBUG    MCTS iteration 500/1500, total nodes: 596
2026-02-25 17:35:15 src.goals.sampling DEBUG    MCTS iteration 550/1500, total nodes: 635
2026-02-25 

eval: {'green': 15, 'orange': 9, 'red': 36, 'total': 60} | 9 unique green completions

... (9 outputs left out)

sample 0 | prefill=''     | {'green': 42, 'orange': 10, 'red': 3, 'total': 55} | 23 green

sample 1 | prefill=''     | {'green': 15, 'orange': 9, 'red': 36, 'total': 60} | 9 green

sample 2 | prefill=''     | {'green': 70, 'orange': 2, 'red': 1, 'total': 73} | 37 green

sample 3 | prefill=''     | {'green': 14, 'orange': 0, 'red': 5, 'total': 19} | 8 green

## Check a Tree to make sure it looks right

In [13]:
rich_print(target_predicate.samples[SAMPLE_IDX])

MultiPredicateSample(
    instruction='Write a Python function that returns True iff the list has no duplicate elements and is sorted in 
non-decreasing order.\n\n',
    code='def is_sorted_unique(lst: list) -> bool:\n    """Return True iff lst has no duplicate elements and is 
sorted in non-decreasing order."""\n    return ',
    unit_tests=[
        ([1, 2, 3], True),
        ([1, 1, 2], False),
        ([3, 2, 1], False),
        ([3, 1, 2], False),
        ([], True),
        ([1], True),
        ([0, 2, 4], True),
        ([1, 2, 2, 3], False),
        ([-1, 0, 1], True)
    ],
    prefills={
        'aligned_p_plus': ['sorted(', 'lst ==', 'lst == sorted('],
        'aligned_p_minus': ['set(lst)', 'len(set(', 'set('],
        'adversarial': ['not (', 'False if', 'True if not'],
        'inconsequential': ['']
    },
    function_name='is_sorted_unique',
    p_minus=PredicateSpec(
        description='check whether len(set(lst)) == len(lst) (all elements unique)',
        anchor_tokens=[
            '== len',
            '== len(lst',
            '== len(lst)',
            'duplicat',
            'len(lst) ==',
            'len(set',
            'len(set(',
            'set(',
            'set(lst)'
        ]
    )
)

In [9]:
from src.goals.sampling import print_tree

# Pick which sample / prefill to inspect
SAMPLE_IDX = 2
PREFILL_IDX = 0

sr   = tp_results.sample_results[SAMPLE_IDX]
run  = sr.runs[PREFILL_IDX]           # first (and currently only) prefill run
root = sr._trees.get(run.prefix)   # None if loaded from disk — need to re-run

if root is None:
    rich_print("[yellow]Tree not in memory (loaded from disk). Re-run the MCTS cell to populate _trees.[/yellow]")
else:
    rich_print(
        f"[bold]Tree[/bold]  sample={SAMPLE_IDX}  fn=[cyan]{sr.sample.function_name}[/cyan]"
        f"  prefill={run.prefill!r}  eval={run.eval_counts}"
    )
    print_tree(root, show_colors=True)

Tree  sample=2  fn=is_sorted_unique  prefill=''  eval={'green': 70, 'orange': 2, 'red': 1, 'total': 73}

[ROOT] visits=1965
    ├── ' all' (p=0.715, v=1425)
    │   ├── '(lst' (p=0.494, v=675)
    │   │   ├── '[i' (p=1.000, v=674)
    │   │   │   ├── ']' (p=0.980, v=647)
    │   │   │   │   ├── ' <' (p=0.918, v=597)
    │   │   │   │   │   └── ' lst' (p=1.000, v=596)
    │   │   │   │   │       └── '[i' (p=1.000, v=595)
    │   │   │   │   │           ├── ' +' (p=0.836, v=490)
    │   │   │   │   │           │   └── ' ' (p=1.000, v=489)
    │   │   │   │   │           │       └── '1' (p=1.000, v=488)
    │   │   │   │   │           │           └── ']' (p=1.000, v=487)
    │   │   │   │   │           │               └── ' for' (p=1.000, v=486)
    │   │   │   │   │           │                   └── ' i' (p=1.000, v=485)
    │   │   │   │   │           │                       └── ' in' (p=1.000, v=484)
    │   │   │   │   │           │                           └── ' range' (p=1.000, v=483)
    │   │   │   │   │           │                               └── '(len' (p=1.000, v=482)
    │   │   │   │   │           │                                   └── '(lst' (p=1.000, v=481)
    │   │   │   │   │           │                                       └── ')' (p=1.000, v=480)
    │   │   │   │   │           │                                           └── ' -' (p=1.000, v=479)
    │   │   │   │   │           │                                               └── ' ' (p=1.000, v=478)
    │   │   │   │   │           │                                                   └── '1' (p=1.000, v=477)
    │   │   │   │   │           │                                                       ├── '))\n' (p=0.527, v=251) [TERMINAL] (' all(lst[i] < lst[i + 1] for i in range(len(lst) - 1))\n')
    │   │   │   │   │           │                                                       ├── '))\n\n' (p=0.320, v=153) [TERMINAL] (' all(lst[i] < lst[i + 1] for i in range(len(lst) - 1))\n\n')
    │   │   │   │   │           │                                                       └── '))' (p=0.151, v=72)
    │   │   │   │   │           │                                                           ├── ' if' (p=0.527, v=38)
    │   │   │   │   │           │                                                           │   ├── ' len' (p=0.797, v=30)
    │   │   │   │   │           │                                                           │   │   ├── '(lst' (p=0.945, v=22)
    │   │   │   │   │           │                                                           │   │   │   └── ')' (p=1.000, v=21)
    │   │   │   │   │           │                                                           │   │   │       └── ' >' (p=1.000, v=20)
    │   │   │   │   │           │                                                           │   │   │           └── ' ' (p=1.000, v=19)
    │   │   │   │   │           │                                                           │   │   │               └── '1' (p=1.000, v=18)
    │   │   │   │   │           │                                                           │   │   │                   └── ' else' (p=1.000, v=17)
    │   │   │   │   │           │                                                           │   │   │                       └── ' True' (p=1.000, v=16)
    │   │   │   │   │           │                                                           │   │   │                           ├── '\n\n' (p=0.703, v=11) [TERMINAL] (' all(lst[i] < lst[i + 1] for i in range(len(lst) - 1)) if len(lst) > 1 else True\n\n')
    │   │   │   │   │           │                                                           │   │   │                           └── '\n' (p=0.293, v=4) [TERMINAL] (' all(lst[i] < lst[i + 1] for i in range(len(lst) - 1)) if len(lst) > 1 else True\n')
    │   │   │   │   │           │                                                           │   │   └── '(set' (p=0.053, v=7)
    │   │   │   │   │           │                                                           │   │       └── '(lst' (p=1.000, v=7)
    │   │   │   │   │           │        

In [11]:
def get_green_solutions_by_first_token(root, prefix):
    """Group green solutions by their first token (reveals which predicate was pursued first)."""
    from collections import defaultdict
    groups = defaultdict(list)
    def walk(node):
        if node.is_leaf() and node.color == "green":
            # Walk up to find the root's direct child (first token)
            n = node
            while n.parent and not n.parent.is_root():
                n = n.parent
            first_tok = n.token_str
            groups[first_tok].append(prefix + node.get_path_text())
        for child in node.children:
            walk(child)
    walk(root)
    return dict(groups)

groups = get_green_solutions_by_first_token(root, prefix="")
for first_tok, codes in groups.items():
    print(f"First token: {first_tok!r}")
    for code in codes:
        print(f'"{repr(code)}"')
        # print("---")

First token: ' all'
"' all(lst[i] < lst[i + 1] for i in range(len(lst) - 1))\n'"
"' all(lst[i] < lst[i + 1] for i in range(len(lst) - 1))\n\n'"
"' all(lst[i] < lst[i + 1] for i in range(len(lst) - 1)) if len(lst) > 1 else True\n\n'"
"' all(lst[i] < lst[i + 1] for i in range(len(lst) - 1)) if len(lst) > 1 else True\n'"
"' all(lst[i] < lst[i + 1] for i in range(len(lst) - 1)) if lst else True\n\n'"
"' all(lst[i] < lst[i + 1] for i in range(len(lst) - 1)) if lst else True\n'"
"' all(lst[i] < lst[i + 1] for i in range(len(lst) - 1)) and len(lst) == len(set(lst))\n\n'"
"' all(lst[i] < lst[i + 1] for i in range(len(lst) - 1)) and len(lst) == len(set(lst))\n'"
"' all(lst[i] < lst[i + 1] for i in range(len(lst) - 1)) and len(set(lst)) == len(lst)\n\n'"
"' all(lst[i] < lst[i + 1] for i in range(len(lst) - 1)) and len(set(lst)) == len(lst)\n'"
"' all(lst[i] < lst[i+1] for i in range(len(lst) - 1))\n'"
"' all(lst[i] < lst[i+1] for i in range(len(lst) - 1))\n\n'"
"' all(lst[i] < lst[i+1] for i in 

## Anchor Extration with Oracle LLMs

In [14]:
from src.goals.anchor_extraction import extract_anchor_tokens

# ── Run oracle extraction ──────────────────────────────────────────────────
# tp_results.completions_per_sample is list[dict[prefix → [completions]]]
# one entry per sample, matching the order of target_predicate.samples
p_plus_anchors, p_minus_anchors_per_sample = extract_anchor_tokens(
    tp=target_predicate,
    completions_per_sample=tp_results.completions_per_sample,
    use_cache=True,
)

# ── Inspect ────────────────────────────────────────────────────────────────
rich_print(f"\n[bold]P+ anchors[/bold] ([cyan]{target_predicate.p_plus.description}[/cyan]):")
rich_print(p_plus_anchors)

for i, (sample, p_minus_anchors) in enumerate(
    zip(target_predicate.samples, p_minus_anchors_per_sample)
):
    p_minus_desc = sample.p_minus.description if sample.p_minus else "?"
    rich_print(f"\n[bold]P- anchors  sample {i}[/bold] ([magenta]{p_minus_desc}[/magenta]):")
    rich_print(p_minus_anchors)

2026-02-25 17:51:20 anthropic._base_client DEBUG    Request options: {'method': 'post', 'url': '/v1/messages', 'timeout': Timeout(connect=5.0, read=600, write=600, pool=600), 'files': None, 'idempotency_key': 'stainless-python-retry-a82b0dbd-a8cb-4e84-8c04-7b7e2b5de2b0', 'content': None, 'json_data': {'max_tokens': 256, 'messages': [{'role': 'user', 'content': [{'type': 'text', 'text': 'You are labeling branches in a code-generation search tree. A language model is completing a Python function stub that must implement two independent boolean predicates:\n\n- **P+** (target predicate): check whether lst is sorted in non-decreasing order (lst == sorted(lst))\n- **P-** (companion predicate): check whether len(lst) >= 2\n\nFunction stub:\n```python\ndef is_long_sorted(lst: list) -> bool:\n    """Return True iff lst has at least 2 elements and is sorted in non-decreasing order."""\n    return\n```\n\nOne valid green completion (passes all unit tests):\n```\nreturn 1 < len(lst) <= len(lst) a

IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out


2026-02-25 17:51:56 httpcore.http11 DEBUG    receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Wed, 25 Feb 2026 22:51:56 GMT'), (b'Content-Type', b'application/json'), (b'Transfer-Encoding', b'chunked'), (b'Connection', b'keep-alive'), (b'anthropic-ratelimit-input-tokens-limit', b'2000000'), (b'anthropic-ratelimit-input-tokens-remaining', b'2000000'), (b'anthropic-ratelimit-input-tokens-reset', b'2026-02-25T22:51:55Z'), (b'anthropic-ratelimit-output-tokens-limit', b'400000'), (b'anthropic-ratelimit-output-tokens-remaining', b'400000'), (b'anthropic-ratelimit-output-tokens-reset', b'2026-02-25T22:51:56Z'), (b'anthropic-ratelimit-requests-limit', b'4000'), (b'anthropic-ratelimit-requests-remaining', b'3999'), (b'anthropic-ratelimit-requests-reset', b'2026-02-25T22:51:55Z'), (b'anthropic-ratelimit-tokens-limit', b'2400000'), (b'anthropic-ratelimit-tokens-remaining', b'2400000'), (b'anthropic-ratelimit-tokens-reset', b'2026-02-25T22:51:55Z'), (b'request-

IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out


2026-02-25 17:53:03 httpcore.http11 DEBUG    receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Wed, 25 Feb 2026 22:53:03 GMT'), (b'Content-Type', b'application/json'), (b'Transfer-Encoding', b'chunked'), (b'Connection', b'keep-alive'), (b'anthropic-ratelimit-input-tokens-limit', b'2000000'), (b'anthropic-ratelimit-input-tokens-remaining', b'2000000'), (b'anthropic-ratelimit-input-tokens-reset', b'2026-02-25T22:53:02Z'), (b'anthropic-ratelimit-output-tokens-limit', b'400000'), (b'anthropic-ratelimit-output-tokens-remaining', b'400000'), (b'anthropic-ratelimit-output-tokens-reset', b'2026-02-25T22:53:03Z'), (b'anthropic-ratelimit-requests-limit', b'4000'), (b'anthropic-ratelimit-requests-remaining', b'3999'), (b'anthropic-ratelimit-requests-reset', b'2026-02-25T22:53:01Z'), (b'anthropic-ratelimit-tokens-limit', b'2400000'), (b'anthropic-ratelimit-tokens-remaining', b'2400000'), (b'anthropic-ratelimit-tokens-reset', b'2026-02-25T22:53:02Z'), (b'request-

IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out


... (8 outputs left out)

P- anchors  sample 2 (check whether len(set(lst)) == len(lst) (all elements unique)):

['== len', '== len(lst', '== len(lst)', 'duplicat', 'len(lst) ==', 'len(set', 'len(set(', 'set(', 'set(lst)']

P- anchors  sample 3 (check whether all(isinstance(x, int) for x in lst)):

['int) for', 'isinstance', 'isinstance(i, int)', 'isinstance(x, int)']

In [15]:
# ── Set anchors on target_predicate and save to disk ──────────────────────
# Review the printed anchors above, edit the lists if needed, then run this cell.

target_predicate.p_plus.anchor_tokens = p_plus_anchors

for i, sample in enumerate(target_predicate.samples):
    if sample.p_minus and p_minus_anchors_per_sample[i]:
        sample.p_minus.anchor_tokens = p_minus_anchors_per_sample[i]

target_predicate.save()   # overwrites data/multi_predicate/<name>.json

rich_print(f"[green]Saved updated anchor tokens → {target_predicate._source_path}[/green]")

2026-02-25 17:56:18 src.goals.predicate_data INFO     Saved TargetPredicate 'is_sorted' to /disk/u/arnab/Codes/Projects/goals/data/multi_predicate/is_sorted.json


Saved updated anchor tokens → /disk/u/arnab/Codes/Projects/goals/data/multi_predicate/is_sorted.json

## Checking contrasting sampling

In [ ]:
from src.goals.probe import (
    compute_outcome_probs,
    sample_contrastive_pairs,
    sample_multidepth_pairs,
    print_contrastive_pairs,
    summarize_contrastive_pairs,
)

# Sample contrastive pairs from the evaluated MCTS tree
# The tree should already be evaluated (search.root has colors assigned)
pairs_by_depth = sample_multidepth_pairs(
    root=search.root,
    min_depth=1,
    max_depth=16,
    min_green_confidence=0.5,
    max_red_in_positive=0.3,  # Slightly relaxed
    min_red_confidence=0.5,
    max_green_in_negative=0.1,
    max_uncertain=0.5,
)

print_contrastive_pairs(pairs_by_depth, max_pairs_per_depth=5)

In [ ]:
# Inspect outcome probabilities at a specific depth
depth_to_inspect = 3
from src.goals.probe import collect_nodes_at_depth

nodes = collect_nodes_at_depth(search.root, depth_to_inspect)
print(f"Nodes at depth {depth_to_inspect}: {len(nodes)}")

for node in nodes[:10]:  # Show first 10
    if not node.is_leaf() and not node.is_terminal:
        probs = compute_outcome_probs(node)
        path_text = node.get_path_text()[:40]
        print(f"  '{path_text}...' -> {probs}")

In [ ]:
# Look at a specific contrastive pair in detail
if pairs_by_depth:
    # Get pairs from the first depth that has them
    first_depth = min(pairs_by_depth.keys())
    pairs = pairs_by_depth[first_depth]
    
    if pairs:
        pair = pairs[0]
        print(f"[bold]Example Contrastive Pair at depth {first_depth}:[/bold]")
        print()
        print(f"[green]Positive path:[/green] {repr(pair.positive.get_path_text())}")
        print(f"  Outcome probs: {pair.pos_probs}")
        print(f"  Node has {len(pair.positive.children)} children")
        print()
        print(f"[red]Negative path:[/red] {repr(pair.negative.get_path_text())}")
        print(f"  Outcome probs: {pair.neg_probs}")
        print(f"  Node has {len(pair.negative.children)} children")
else:
    print("[yellow]No contrastive pairs found. Try adjusting thresholds or running more MCTS iterations.[/yellow]")